<a href="https://colab.research.google.com/github/matthew-ngzc/AI-Safety-Module/blob/main/Week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone the repository

In [1]:
try:
    ! git clone https://github.com/cs612-smu/cs612-smu-2025 CS612_SMU
    HOME_DIR = "./CS612_SMU/week5/"
except:
    print('Already clone!!!')

Cloning into 'CS612_SMU'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 180 (delta 70), reused 172 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 30.20 MiB | 42.13 MiB/s, done.
Resolving deltas: 100% (70/70), done.


Exercise 2: In this exercise, we experiment with a simple simple MIA attack.

In [ ]:
## CIFAR100 ##
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import random

class CIFAR100Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 100)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def save_model(model, name):
    torch.save(model.state_dict(), name)

def load_model(model_class, name, *args):
    model = model_class(*args)
    model.load_state_dict(torch.load(name, map_location=device))
    return model

def train(model, dataloader, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        # Compute prediction error
        pred = model(x)
        loss = loss_fn(pred, y)
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if batch % 100 == 0:
            loss_val, current = loss.item(), batch * len(x)
            print('loss: {:.4f} [{}/{}]'.format(loss_val, current, size))

def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()
    loss /= num_batches
    correct /= size
    print('Test Error: \n Accuracy: {:.2f}%, Avg loss: {:.4f}'.format(100 * correct, loss))

def attack(model, dataloader, loss_fn, device):
    size = 10000
    num_batches = len(dataloader)
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            pred = model(x)
            #TODO: add one line t update variable correct to evaluate the accuacy of this simple MIA attack
            #Note that this MIA attack predicts a sample is a member iff the prediction is correct.
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

            if (batch + 1) * len(y) == size: break
    return correct / size * 100

train_kwargs = {'batch_size': 1000}
test_kwargs = {'batch_size': 1000}
transform = transforms.ToTensor()

train_dataset = datasets.CIFAR100('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR100('./data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, **train_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)


model = CIFAR100Net().to(device)
model = load_model(CIFAR100Net, HOME_DIR + 'exercise2/cifar100.pt').to(device)

MIAAttackTrain = attack(model, train_loader, nn.CrossEntropyLoss(), device)
MIAAttackTest = 100 - attack(model, test_loader, nn.CrossEntropyLoss(), device)

print('Overall MIA accuracy: {:.2f}%\n'.format((MIAAttackTrain+MIAAttackTest)/2))
print('MIA accuracy on train data: {:.2f}%\n'.format(MIAAttackTrain))
print('MIA accuracy on test data: {:.2f}%\n'.format(MIAAttackTest))

Overall MIA accuracy: 76.67%

MIA accuracy on train data: 89.34%

MIA accuracy on test data: 64.01%



Exercise 4: In this exercise, you will try training with differential privacy.

In [ ]:
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import numpy as np
import math


class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 10)
        self.fc4 = nn.Linear(10, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        output = x # cross entropy in pytorch already includes softmax
        return output


def save_model(model, name):
    torch.save(model.state_dict(), name)


def load_model(model_class, name, *args):
    model = model_class(*args)
    model.load_state_dict(torch.load(name, map_location=torch.device('cpu')))

    return model


def train(model, dataloader, loss_fn, optimizer, device, delta, epsilon):
    sigma = math.sqrt(2 * math.log(1.25 / delta)) / epsilon
    size = len(dataloader.dataset)
    model.train()

    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        # Compute prediction error
        pred = model(x)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        #The following adds the noise according to differential privacy;
        for name, param in model.named_parameters():
            param.grad.data += np.random.normal(loc=0.0, scale=sigma, size=param.grad.data.size()) / len(y)

        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print('loss: {:.4f} [{}/{}]'.format(loss, current, size))


def test(model, dataloader, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    model.eval()
    loss, correct = 0.0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.int).sum().item()

    loss /= num_batches
    correct /= size
    print('Test Error: \n Accuracy: {:.2f}%, Avg loss: {:.4f}\n'.format(100 * correct, loss))

def compute_mentr(pred, y):
    pred, y = pred.numpy(), y.numpy()
    mentr = []

    for i in range(len(y)):
        val = 0.0
        for j in range(len(pred[i])):
            if j == y[i]:
                val -= (1 - pred[i][j]) * math.log(pred[i][j])
            elif pred[i][j] < 1:
                val -= pred[i][j] * math.log(1 - pred[i][j])
        mentr.append(val)

    return np.array(mentr)

def attack(model, dataloader, loss_fn, device, threshold):
    size = 10000
    num_batches = len(dataloader)

    model.eval()
    correct = 0

    with torch.no_grad():
        for batch, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            pred = F.softmax(model(x), 1)
            mentr = compute_mentr(pred, y)
            correct += (mentr < threshold).sum()
            if (batch + 1) * len(y) == size: break

    return correct / size * 100


device = 'cpu'
train_kwargs = {'batch_size': 100}
test_kwargs = {'batch_size': 1000}
transform = transforms.ToTensor()

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, **train_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)

model = MNISTNet().to(device)

optimizer = optim.SGD(model.parameters(), lr=0.1)
num_of_epochs = 20
delta, epsilon = 1e-3, 10

print("A program demonstrating training with differential privacy.")

for epoch in range(num_of_epochs):
   print('\n------------- Epoch {} -------------\n'.format(epoch))
   train(model, train_loader, nn.CrossEntropyLoss(), optimizer, device, delta, epsilon)
   test(model, test_loader, nn.CrossEntropyLoss(), device)

threshold = 0.9

MIAAttackTrain = attack(model, train_loader, nn.CrossEntropyLoss(), device, threshold)
MIAAttackTest = 100 - attack(model, test_loader, nn.CrossEntropyLoss(), device, threshold)

print('Overall MIA accuracy: {:.2f}%\n'.format((MIAAttackTrain+MIAAttackTest)/2))
print('MIA accuracy on train data: {:.2f}%\n'.format(MIAAttackTrain))
print('MIA accuracy on test data: {:.2f}%\n'.format(MIAAttackTest))


A program demonstrating training with differential privacy.

------------- Epoch 0 -------------

loss: 2.3164 [0/60000]


/tmp/ipykernel_729/2770562971.py:68: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  param.grad.data += np.random.normal(loc=0.0, scale=sigma, size=param.grad.data.size()) / len(y)


loss: 2.0669 [10000/60000]
loss: 1.5024 [20000/60000]
loss: 1.1804 [30000/60000]
loss: 0.7290 [40000/60000]
loss: 0.5514 [50000/60000]
Test Error: 
 Accuracy: 84.23%, Avg loss: 0.5232


------------- Epoch 1 -------------

loss: 0.4948 [0/60000]
loss: 0.5182 [10000/60000]
loss: 1.0835 [20000/60000]
loss: 0.3807 [30000/60000]
loss: 0.3847 [40000/60000]
loss: 0.3689 [50000/60000]
Test Error: 
 Accuracy: 89.93%, Avg loss: 0.3377


------------- Epoch 2 -------------

loss: 0.2010 [0/60000]
loss: 0.4364 [10000/60000]
loss: 0.4435 [20000/60000]
loss: 0.2889 [30000/60000]
loss: 0.3182 [40000/60000]
loss: 0.3062 [50000/60000]
Test Error: 
 Accuracy: 90.86%, Avg loss: 0.2977


------------- Epoch 3 -------------

loss: 0.1617 [0/60000]
loss: 0.4078 [10000/60000]
loss: 0.3453 [20000/60000]
loss: 0.2491 [30000/60000]
loss: 0.2710 [40000/60000]
loss: 0.3473 [50000/60000]
Test Error: 
 Accuracy: 92.00%, Avg loss: 0.2647


------------- Epoch 4 -------------

loss: 0.1512 [0/60000]
loss: 0.3389 [10